In [2]:
import pandas as pd
import numpy as np
import random
from nltk.corpus import wordnet

In [3]:
df = pd.read_csv('../data/news_dataset.csv', encoding='utf-8-sig')

In [4]:
df['label'].value_counts()

label
Legal news and crime                          53
Tourism and travel news                       47
Education and schools                         32
Political news and government affairs         32
Health, Medical and Sports                    30
Disaster management and natural calamities    26
Agriculture news and updates                  14
Infrastructure projects and development       14
Environment and ecology                       11
Economic and financial news                    9
Name: count, dtype: int64

In [5]:
# EDA helper functions
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lm in syn.lemmas():
            synonym = lm.name().replace('_', ' ').lower()
            if synonym != word:
                synonyms.add(synonym)
    return list(synonyms)

In [6]:
def synonym_replacement(words, n=1):
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word.isalpha()]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word ==
                         random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return new_words

In [7]:
def random_deletion(words, p=0.1):
    if len(words) == 1:
        return words
    new_words = []
    for word in words:
        r = random.uniform(0, 1)
        if r > p:
            new_words.append(word)
    if len(new_words) == 0:
        return [words[random.randint(0, len(words)-1)]]
    return new_words

In [8]:
def random_swap(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(new_words)), 2)
        new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    return new_words

In [9]:
def random_insertion(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        add_random_word(new_words)
    return new_words

In [10]:
def add_random_word(words):
    synonyms = []
    counter = 0
    while len(synonyms) < 1 and counter < 10:
        random_word = words[random.randint(0, len(words)-1)]
        synonyms = get_synonyms(random_word)
        counter += 1
    if len(synonyms) >= 1:
        random_synonym = random.choice(synonyms)
        random_idx = random.randint(0, len(words)-1)
        words.insert(random_idx, random_synonym)

In [11]:
def eda_augment(text, num_aug=1):
    words = text.split()
    augmented_sentences = []
    for _ in range(num_aug):
        choice = random.choice(['sr', 'rd', 'rs', 'ri'])  # 4 methods
        if choice == 'sr':
            new_words = synonym_replacement(words, n=1)
        elif choice == 'rd':
            new_words = random_deletion(words, p=0.1)
        elif choice == 'rs':
            new_words = random_swap(words, n=1)
        else:
            new_words = random_insertion(words, n=1)
        augmented_sentences.append(' '.join(new_words))
    return augmented_sentences

In [12]:
# Augment minority classes
target_size = 50
new_rows = []

for label, group in df.groupby('label'):
    current_count = len(group)
    if current_count < target_size:
        needed = target_size - current_count
        samples = group.sample(needed, replace=True, random_state=42)
        for _, row in samples.iterrows():
            aug_text = eda_augment(row['title'], num_aug=1)[0]
            new_rows.append({'title': aug_text, 'label': label})

aug_df = pd.DataFrame(new_rows)
final_df = pd.concat([df, aug_df], ignore_index=True)

# Shuffle and save
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [13]:
final_df['label'].value_counts()

label
Legal news and crime                          53
Agriculture news and updates                  50
Education and schools                         50
Health, Medical and Sports                    50
Environment and ecology                       50
Infrastructure projects and development       50
Tourism and travel news                       50
Political news and government affairs         50
Disaster management and natural calamities    50
Economic and financial news                   50
Name: count, dtype: int64

In [ ]:
final_df.to_csv('../data/news_dataset_balanced.csv',
                index=False, encoding='utf-8')